In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

SEED = 42
np.random.seed(SEED)

# Загружаем очищенные данные
df = pd.read_csv('../data/processed/cleaned_data.csv')

# Определяем целевую переменную (возможно, там уже 'order_total')
target_col = 'order_total' if 'order_total' in df.columns else 'total_spend'
X = df.drop(columns=[target_col, 'total_spent_log'])
y = df[target_col]

# Разделяем на train / val / test (60/20/20)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=SEED)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (60000, 23), Val: (20000, 23), Test: (20000, 23)


In [5]:
# Сохраняем сплит (опционально)
for name, data in zip(['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test'],
                      [X_train, X_val, X_test, y_train, y_val, y_test]):
    data.to_csv(f'../data/processed/{name}.csv', index=False)

# Определяем числовые и категориальные колонки
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['category', 'object']).columns.tolist()

# Убираем из числовых те, которые на самом деле категориальные (например, hour, day_of_week)
# оставим их как числовые (можно и как категории, но для простоты оставим)
# Создаём препроцессор
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])

# Сохраняем препроцессор (понадобится позже)
import joblib
joblib.dump(preprocessor, '../models/preprocessor.pkl')

/var/folders/t9/w50p8w0917b0t61ks_6y_3g40000gn/T/ipykernel_24664/3581940995.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['category', 'object']).columns.tolist()


['../models/preprocessor.pkl']